# Generative AI Application Deployment and Monitoring

## Overview

- Prepare dataset for a summarization task
- Create a pipeline with HuggingFace for summarization
    - Use model registry and versioned models with MLFlow
- Perform batch inference with different approaches: single-node, multi-node, SQL/`ai_query`()


## Initial setup

In [0]:
%sql
use catalog `studies`;
use schema `databricks-dev`;

In [0]:
%pip freeze | grep transformers

transformers==4.36.2


In [0]:
import warnings
warnings.filterwarnings('ignore')
from rich import print
import json
import tqdm
import pandas as pd
from langchain_community.chat_models import ChatDatabricks
from langchain_core.messages import SystemMessage, HumanMessage
import mlflow
from transformers import pipeline
from delta.tables import DeltaTable
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

In [0]:
JSON_WRITER_SUMMARIES_PATH = '/Volumes/studies/databricks-dev/data_json/writer_summaries.json'

SERVING_MODELS = {
    'gpt-5-1': 'databricks-gpt-5-1',  # disabled
    'gpt-oss-20b': 'databricks-gpt-oss-20b',  # disabled
    'meta-llama-8b': 'databricks-meta-llama-3-1-8b-instruct',  # enabled
    'qwen-80b': 'databricks-qwen3-next-80b-a3b-instruct',  # enabled
    'llama-maverick-400b': 'databricks-llama-4-maverick', # enabled
    'gemma-12b': 'databricks-gemma-3-12b'  # enabled  
}

EXPORT_TABLE = False

## Prepare dataset

In [0]:
def load_json(json_name):
    with open(json_name, 'r', encoding='utf-8') as f:
        data = json.load(f)
        return data


def pandas_to_databricks_table(table_name: str, pandas_df: pd.DataFrame) -> None:
    spark_df = spark.createDataFrame(pandas_df)
    spark_df.write.format('delta').mode('overwrite').saveAsTable(table_name)
    print(spark_df.count)
    display(spark_df.show(7))

In [0]:
df_writer_summaries = pd.DataFrame(load_json(JSON_WRITER_SUMMARIES_PATH))
df_writer_summaries.rename(columns={'summary': 'human_summary'}, inplace=True)
print(df_writer_summaries.shape)
df_writer_summaries.head()

(302, 3)

,article_id,article,human_summary
0,0adb86356834452298d180104ff54179,Nick Scholfield is lined up to ride Spring Hee...,Nick Schofield is riding Spring Heeled in the ...
1,0adb86356834452298d180104ff54179,Nick Scholfield is lined up to ride Spring Hee...,Preparation is taking place for a horse ridin...
2,0adb86356834452298d180104ff54179,Nick Scholfield is lined up to ride Spring Hee...,Nick Scholfield will travel to Ireland and is ...
3,b3168ab4857d4190ac3b2eb46d096f81,Dr Mehmet Oz's fellow faculty members at Colum...,Faculty at Columbia University have written an...
4,b3168ab4857d4190ac3b2eb46d096f81,Dr Mehmet Oz's fellow faculty members at Colum...,"Celebrity doctor, Mehmet Oz, is being attacked..."


In [0]:
df_writer_summaries['len_article'] = df_writer_summaries.article.apply(lambda x: len(x))

In [0]:
df_writer_summaries.describe()

,len_article
count,302.000000
mean,4116.913907
std,1870.479140
min,661.000000
25%,2606.000000
50%,3944.000000
75%,5139.000000
max,8978.000000


In [0]:
df_writer_summaries[df_writer_summaries.len_article < 1000].shape

(8, 4)

In [0]:
one_sample = df_writer_summaries[df_writer_summaries.len_article < 1000].iloc[0].to_dict()
print(f'** Length of Article:** {one_sample['len_article']}')
print(f'**Article:**\n{one_sample['article']}')
print(f'**Summary:**\n{one_sample['human_summary']}')

** Length of Article:** 661

**Article:**
(CNN) -- A high-speed passenger train left its tracks on the outskirts of Split, Croatia, Friday, killing at least 
six people and injuring 45, according to Croatian police. The high-speed train derailed on the outskirts of Split, 
Croatia, about noon on Friday. The train was on its way from the Croatian capital, Zagreb, when it derailed about 
20 kilometers (12 miles) from it's destination of Split about noon, said Marina Kraljevic-Gudelj, a spokeswoman for
police in Split. "This is a huge tragedy, so there is no place for speculation," she said. Police had launched an 
investigation into the cause of the crash. CNN's Per Nyberg contributed to this report.

**Summary:**
A high-speed passenger train derailed on Friday, killing six and injuring 45. Police have launched an investigation
into the cause of the crash which occurred on the outskirts of Split, Croatia.

In [0]:
if EXPORT_TABLE:
    pandas_to_databricks_table(table_name='writer_summaries', pandas_df=df_writer_summaries)

In [0]:
%sql
select * from writer_summaries
limit 5;

article_id article human_summary len_article 0adb86356834452298d180104ff54179 Nick Scholfield is lined up to ride Spring Heeled in the Grand National at Aintree on April 11.

Nick Scholfield has been lined up to ride Jim Culloty’s Spring Heeled in the Crabbie’s Grand National at Aintree on Saturday week.

Scholfield had been expected to partner Paul Nicholls-trained Sam Winner, who was pulled up in the Cheltenham Gold Cup, in the £1million race.

But the champion trainer said on Wednesday it was unfair to tie Scholfield down to a gelding which is far from certain to run when the mount on another leading definite contender is being offered.

Scholfield, who has ridden in six Nationals and finished third in 2013 on Teaforthree, will travel to Ireland to sit on Spring Heeled at Culloty’s County Cork stable on Friday.

Nicholls said: ‘I have not made up my mind if I am going to run Sam Winner yet and Nick needed a decision.

‘I did not want to get into a situation next week when I had to say "sorry mate, he is not running" and did not want to stop him getting a good ride.

‘I have not pressed any buttons on any of the horses who ran at Cheltenham. That will happen over the weekend and early next week. I don’t want to run unless I am really happy.

‘I have plenty of other lads who could ride Sam Winner if he runs and would not be afraid to use Will Biddick or Harry Skelton.’

Spring Heeled (right) wins the Fulke Walwyn Kim Muir Challenge Cup at Cheltenham last year.

Spring Heeled, winner of Fulke Walwyn Kim Muir Challenge Cup at last season’s Cheltenham Festival, has been given a National preparation.

The eight-year-old has run only once since finishing fourth to Road To Riches in the Galway Plate in July when he was fourth of five in the Bobbyjo Chase at Fairyhouse in February.

Racemail revealed on Wednesday that Culloty would have two runners in the National.

Robbie McNamara will ride his 2014 Gold Cup winner Lord Windermere.

Scholfield rides Teaforthree (front) as the horse jumps the last fence at Aintree in the 2013 Grand National.

McNamara said: ‘It's a great ride to get and I'm looking forward to it. I've ridden him before in a Grade One in Leopardstown and I was supposed to ride him in the Hennessy there as well, but I broke my collarbone the day before. I'm delighted to get back on him.’

With Nigel Twiston-Davies-trained Double Ross another confirmed non runner, David Pipe’s well supported Soll appears guaranteed a run at the bottom of the weights.

Luke Morris became the first jockey to ride 100 winners during an All Weather Flat racing season when a double at Chelmsford on Wednesday aboard Giantouch and Middle East Pearl carried him to 101 successes for the campaign. Nick Schofield is riding Spring Heeled in the Crabbie's Grand National on Saturday. Schofield was expected to ride Sam Winner. Says Schofield, "I have plenty of other lads who could ride Sam Winner..." Spring Heeled has only run once since finishing fourth in the Galway Plate. 2650 0adb86356834452298d180104ff54179 Nick Scholfield is lined up to ride Spring Heeled in the Grand National at Aintree on April 11.

Nick Scholfield has been lined up to ride Jim Culloty’s Spring Heeled in the Crabbie’s Grand National at Aintree on Saturday week.

Scholfield had been expected to partner Paul Nicholls-trained Sam Winner, who was pulled up in the Cheltenham Gold Cup, in the £1million race.

But the champion trainer said on Wednesday it was unfair to tie Scholfield down to a gelding which is far from certain to run when the mount on another leading definite contender is being offered.

Scholfield, who has ridden in six Nationals and finished third in 2013 on Teaforthree, will travel to Ireland to sit on Spring Heeled at Culloty’s County Cork stable on Friday.

Nicholls said: ‘I have not made up my mind if I am going to run Sam Winner yet and Nick needed a decision.

‘I did not want to get into a situation next week when I had to say "sorry mate, he is not running"

## Pipeline with HuggingFace

- Model to summarization: **T5 text-to-text transfer transformer**
- https://huggingface.co/google-t5/t5-small
    - https://huggingface.co/Rahmat82/t5-small-finetuned-summarization-xsum


In [0]:
print(one_sample['article'])

(CNN) -- A high-speed passenger train left its tracks on the outskirts of Split, Croatia, Friday, killing at least 
six people and injuring 45, according to Croatian police. The high-speed train derailed on the outskirts of Split, 
Croatia, about noon on Friday. The train was on its way from the Croatian capital, Zagreb, when it derailed about 
20 kilometers (12 miles) from it's destination of Split about noon, said Marina Kraljevic-Gudelj, a spokeswoman for
police in Split. "This is a huge tragedy, so there is no place for speculation," she said. Police had launched an 
investigation into the cause of the crash. CNN's Per Nyberg contributed to this report.

In [0]:
params_summarizer = {
    'model_id': 'Rahmat82/t5-small-finetuned-summarization-xsum',
    'min_length': 20,
    'max_length': 50,    
    'truncation': True,
    'do_sample': True,
    'device_map': -1  # cpu
}

model = AutoModelForSeq2SeqLM.from_pretrained(params_summarizer['model_id'])
tokenizer = AutoTokenizer.from_pretrained(params_summarizer['model_id'], use_fast=True)

summarizer = pipeline(
    task='summarization',
    model=model, 
    tokenizer=tokenizer,
    min_length=params_summarizer['min_length'],
    max_length=params_summarizer['max_length'],
    truncation=params_summarizer['truncation'],
    do_sample=params_summarizer['do_sample'],
    device_map=params_summarizer['device_map'],
)


res = summarizer(one_sample['article'])[0]['summary_text']
res
print(res)

A train has derailed in the Croatian capital, Split, killing at least six people and injuring more than 100, police
say.

### Model registering with MLFlow

- model/pipelines > experiments > runs > (logs, metrics, artifacts, metadata)

In [0]:
import mlflow
from mlflow.models import infer_signature
from mlflow.transformers import generate_signature_output

In [0]:
# define the signature of input and output schemas

output = generate_signature_output(summarizer, one_sample['article'])
signature = infer_signature(one_sample['article'], output)

print(f'Signature:{signature}')

Signature:inputs: 
  
outputs: 
  
params: 
  None

In [0]:
print(params_summarizer)

{
    'model_id': 'Rahmat82/t5-small-finetuned-summarization-xsum',
    'min_length': 20,
    'max_length': 50,
    'truncation': True,
    'do_sample': True,
    'device_map': -1
}

In [0]:
# define experiment path
# sidebar > machine learning > experiments

batch_size = 1
device = -1  # -1 = cpu
binary_output = False
num_workers = 8

model_artifact_path = 'summarizer_artifact'  # output to searialized model
user_name = spark.sql('SELECT current_user()').collect()[0][0]
experiment_name = f'/Users/{user_name}/experiment-genai-as-batch-demo'
# an experiment name must be an absolute path within the Databricks workspace, e.g. '/Users/<some-username>/my-experiment

mlflow.set_experiment(experiment_name)
with mlflow.start_run():
    # log params
    mlflow.log_params(params_summarizer)

    # log model
    model_info = mlflow.transformers.log_model(
        task='summarization',
        # artifact_path=model_artifact_path,
        name=model_artifact_path,
        transformers_model=summarizer,
        signature=signature,
        inference_config={
            'min_length': params_summarizer['min_length'],
            'max_length': params_summarizer['max_length'],
            'truncation': params_summarizer['truncation'],
            'do_sample': params_summarizer['do_sample'],
            'device_map': params_summarizer['device_map'],
        },       
        input_example=one_sample['article']
    )


🔗 View Logged Model at: https://dbc-34ac7b3c-7a54.cloud.databricks.com/ml/experiments/3717431272157322/models/m-eadbb9c4c3d5498aa1e2bceec18e833a?o=7474656785376246
Your max_length is set to 200, but your input_length is only 167. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=83)


In [0]:
def get_last_summarizer_model_from_experiment(experiment_name):
    # take the last model from tracking server
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
    runs = mlflow.search_runs([experiment_id])
    last_run_id = runs.sort_values('start_time', ascending=False).iloc[0].run_id

    # define model uri from run_id
    model_uri = f'runs:/{last_run_id}/{model_artifact_path}'

    # load the last model tracked
    last_summarizer = mlflow.pyfunc.load_model(model_uri=model_uri)

    return last_summarizer

In [0]:
lastest_summarizer = get_last_summarizer_model_from_experiment(experiment_name)

In [0]:
# summarize using the last model loaded
res_test = lastest_summarizer.predict(one_sample['article'])
print(res_test)

Your max_length is set to 200, but your input_length is only 167. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=83)


[
    'A high-speed train has derailed in Croatia, killing at least six people and injuring more than a dozen others,
police have said.'
]

In [0]:
res

'A train has derailed in the Croatian capital, Split, killing at least six people and injuring more than 100, police say.'

In [0]:
database_name = spark.sql('SELECT current_database()').collect()[0][0]
catalog_name = spark.sql('SELECT current_catalog()').collect()[0][0]
schema_name = spark.sql('SELECT current_schema()').collect()[0][0]


experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
runs = mlflow.search_runs([experiment_id])
last_run_id = runs.sort_values('start_time', ascending=False).iloc[0].run_id
get_last_summarizer_model_from_experiment(experiment_name)
model_uri = f'runs:/{last_run_id}/{model_artifact_path}'

# push model artifact to unity catalog registry
mlflow.set_registry_uri(database_name)
model_name = 'summarizer_test'  #  f'{catalog_name}.{schema_name}.summarizer'
mlflow.register_model(
	model_uri=model_uri,
	name=model_name
)

Registered model 'summarizer_test' already exists. Creating a new version of this model...
2026/02/11 10:00:48 WARNING mlflow.tracking._model_registry.fluent: Run with id 83098a4441e24f6ca879603aa105d957 has no artifacts at artifact path 'summarizer_artifact', registering model based on models:/m-eadbb9c4c3d5498aa1e2bceec18e833a instead
Created version '5' of model 'summarizer_test'.


<ModelVersion: aliases=[], creation_timestamp=1770804048593, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1770804048593, metrics=[], model_id='m-eadbb9c4c3d5498aa1e2bceec18e833a', name='summarizer_test', params={'device_map': '-1',
 'do_sample': 'True',
 'max_length': '50',
 'min_length': '20',
 'model_id': 'Rahmat82/t5-small-finetuned-summarization-xsum',
 'truncation': 'True'}, run_id='83098a4441e24f6ca879603aa105d957', run_link='https://dbc-34ac7b3c-7a54.cloud.databricks.com?o=7474656785376246#mlflow/experiments/3717431272157322/runs/83098a4441e24f6ca879603aa105d957', source='models:/m-eadbb9c4c3d5498aa1e2bceec18e833a', status='READY', status_message=None, tags={}, user_id=None, version=5>

### Set the last model version as @latest

In [0]:
from mlflow.tracking import MlflowClient
client = MlflowClient()

## Get latest model version

In [0]:
versions = client.get_latest_versions(name=model_name)
client.set_registered_model_alias(
	name=model_name,
	alias='latest_update',
	version=versions[0].version
)

In [0]:
model_name = 'summarizer_test'
current_model_version_candidates = []
for v in versions:
    print(f'Version: {v.version}')
    print(f'Stage: {v.current_stage}')
    print(f'Run ID: {v.run_id}')
    if v.run_id:
        current_model_version_candidates.append({'version': v.version, 'stage': v.current_stage})

Version: 5

Stage: None

Run ID: 83098a4441e24f6ca879603aa105d957

In [0]:
current_model_version = current_model_version_candidates[-1]['version']
current_model_stage = current_model_version_candidates[-1]['stage']
print(f'current_model_version: {current_model_version}, current_model_stage: {current_model_stage}')

latest_model = mlflow.pyfunc.load_model(
	model_uri=f'models:/{model_name}/{current_model_version}',
    
)
latest_model

current_model_version: 5, current_model_stage: None

mlflow.pyfunc.loaded_model:
  artifact_path: dbfs:/databricks/mlflow-tracking/3717431272157322/logged_models/m-eadbb9c4c3d5498aa1e2bceec18e833a/artifacts
  flavor: mlflow.transformers
  run_id: 83098a4441e24f6ca879603aa105d957

In [0]:
N_MAX = 2
sample_df = df_writer_summaries[df_writer_summaries.len_article < 1000].head(N_MAX)

## Single-node batch inference

In [0]:
pred_summaries = latest_model.predict(sample_df.article)
print(f'\nArticles:\n{sample_df.article.tolist()}')
print(f'\nPredicted Summaries:\n{pred_summaries}')
print(pred_summaries)

Your max_length is set to 200, but your input_length is only 167. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=83)
Your max_length is set to 200, but your input_length is only 178. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=89)


Articles:
['(CNN) -- A high-speed passenger train left its tracks on the outskirts of Split, Croatia, Friday, killing at 
least six people and injuring 45, according to Croatian police. The high-speed train derailed on the outskirts of 
Split, Croatia, about noon on Friday. The train was on its way from the Croatian capital, Zagreb, when it derailed 
about 20 kilometers (12 miles) from it\'s destination of Split about noon, said Marina Kraljevic-Gudelj, a 
spokeswoman for police in Split. "This is a huge tragedy, so there is no place for speculation," she said. Police 
had launched an investigation into the cause of the crash. CNN\'s Per Nyberg contributed to this report.', 
"BAGHDAD, Iraq (CNN) -- Six gay men were shot dead by members of their tribe in two separate incidents in the past 
10 days, an official with Iraq's Interior ministry said. In the most recent attack, two men were killed Thursday in
Sadr City area of Baghdad after they were disowned by relatives, the official said. The shootings came after a 
tribal meeting was held and the members decided to go after the victims. On March 26, four additional men were 
fatally shot in the same city, the official said, adding that the victims had also been disowned by their 
relatives. The official declined to be identified because he is not authorized to speak to the media. Witnesses 
told CNN that a Sadr City cafe, which was a popular gathering spot for gays, was also set on fire."]

Predicted Summaries:
['A high-speed train has derailed in Croatia, killing at least six people and injuring more than a dozen others, 
police have said.', 'Six gay men have been shot dead in the Iraqi city of Baghdad, officials say, after a tribal 
meeting was held in the city.']

[
    'A high-speed train has derailed in Croatia, killing at least six people and injuring more than a dozen others,
police have said.',
    'Six gay men have been shot dead in the Iraqi city of Baghdad, officials say, after a tribal meeting was held 
in the city.'
]

## Scaling batch inference - multi-node batch inference with `spark_udf`

In [0]:
model_udf = mlflow.pyfunc.spark_udf(
	spark,
 	model_uri=f'models:/{model_name}/{current_model_version}',  # model_uri=f'models:/{model_name}/@latest_update',
 	env_manager='local',
	result_type='string'
)

2026/02/11 10:01:12 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2026/02/11 10:01:13 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


In [0]:
sample_spark = spark.createDataFrame(sample_df) 
sample_spark.show()

+--------------------+--------------------+--------------------+-----------+
|          article_id|             article|       human_summary|len_article|
+--------------------+--------------------+--------------------+-----------+
|fbb2c1430c3548d48...|(CNN) -- A high-s...|A high-speed pass...|        661|
|2e5837f2f9e440d0b...|BAGHDAD, Iraq (CN...|Six gay men were ...|        768|
+--------------------+--------------------+--------------------+-----------+



In [0]:
# from pyspark.sql import functions as F
# batch_inference_res = sample_spark.withColumn('predicted_summary', model_udf(F.col('article')))
# batch_inference_res.show(truncate=False)

In [0]:
# display(batch_inference_res)
# Note:  Needs configure a compatible cluster for distributed inference.

In [0]:
# summaries_table_name = f'{catalog_name}.{schema_name}.multinode_batch_inference'
# batch_inference_res.write.mode('append').saveAsTable(summaries_table_name)

## Batch inference with `ai_query`

In [0]:
SERVING_MODELS['meta-llama-8b']

'databricks-meta-llama-3-1-8b-instruct'

In [0]:
%sql
create or replace table ai_query_inference as(
	select
	article_id, 
    ai_query(
		'databricks-meta-llama-3-1-8b-instruct',
		concat('Given a text input of a article, you need to return a summary in only one sentence. Column of article:', article) 
	) as predicted_summary
	from writer_summaries limit 5
)


num_affected_rows,num_inserted_rows


In [0]:
%sql
select distinct *
from ai_query_inference;

article_id,predicted_summary
0adb86356834452298d180104ff54179,"Here is a one-sentence summary of the article: Nick Scholfield has been lined up to ride Spring Heeled in the Crabbie's Grand National at Aintree on April 11, after initially being expected to ride Sam Winner."
b3168ab4857d4190ac3b2eb46d096f81,"Here is a one-sentence summary of the article: Dr. Mehmet Oz's colleagues at Columbia University have written an op-ed defending him against criticism from 10 doctors who accused him of promoting ""quack treatments"" on his TV show, while also acknowledging that his on-air advice can be ""unsubstantiated"" and potentially harm patients."
